In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Question 1

In [2]:
orders = pd.read_csv('orders.csv')
orders['order_date'] = pd.to_datetime(orders['order_date'])
counts = orders['customer_id'].value_counts()
multiple_counts = counts[counts > 1].index.tolist()

filtered_orders = orders[orders['customer_id'].isin(multiple_counts)].copy()

# 2. Sort by customer AND date so consecutive rows are in chronological order
filtered_orders = filtered_orders.sort_values(['customer_id', 'order_date'])

# 3. Calculate the gap (difference between current and previous row within each group)
filtered_orders['inter_order_gap'] = filtered_orders.groupby('customer_id')['order_date'].diff().dt.days

print(f'Trung vị số ngày giữa hai lần mua liên tiếp: {filtered_orders['inter_order_gap'].median()}')

Trung vị số ngày giữa hai lần mua liên tiếp: 144.0


Đáp án: C) 180 ngày

---

## Question 2

In [3]:
products = pd.read_csv('products.csv')
products = products[products['price'] > 0]
products['gross_profit_margin'] = (products['price'] - products['cogs'])/products['price']
average_gpm = products.groupby('segment')['gross_profit_margin'].mean()

print(average_gpm)
print('-'*40)
print(f'Phân khúc sản phẩm có tỉ suất lợi nhuận gộp trung bình cao nhất: {average_gpm.idxmax()}')

segment
Activewear     0.265600
All-weather    0.284176
Balanced       0.258038
Everyday       0.236343
Performance    0.263650
Premium        0.285377
Standard       0.313442
Trendy         0.240758
Name: gross_profit_margin, dtype: float64
----------------------------------------
Phân khúc sản phẩm có tỉ suất lợi nhuận gộp trung bình cao nhất: Standard


Đáp án: D) Standard

---

## Question 3

In [4]:
returns = pd.read_csv('returns.csv')
prod_return_join = pd.merge(products, returns, on='product_id', how = 'inner')
prod_return_join_streetwear = prod_return_join[prod_return_join['category']=='Streetwear'].copy()

return_reason = prod_return_join_streetwear['return_reason'].value_counts()

print(return_reason)
print('-'*40)
print(f'Lí do trả hàng xuất hiện nhiều nhất: {return_reason.idxmax()}')

return_reason
wrong_size          7626
defective           4330
not_as_described    3854
changed_mind        3830
late_delivery       2159
Name: count, dtype: int64
----------------------------------------
Lí do trả hàng xuất hiện nhiều nhất: wrong_size


Đáp án: B) wrong_size

---

## Question 4

In [5]:
web_traffic = pd.read_csv('web_traffic.csv')

average_bounce_rate = web_traffic.groupby('traffic_source')['bounce_rate'].mean()

print(average_bounce_rate)
print('-'*40)
print(f'Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất: {average_bounce_rate.idxmin()}')

traffic_source
direct            0.004511
email_campaign    0.004458
organic_search    0.004504
paid_search       0.004478
referral          0.004499
social_media      0.004476
Name: bounce_rate, dtype: float64
----------------------------------------
Nguồn truy cập có tỷ lệ thoát trung bình thấp nhất: email_campaign


Đáp án: C) email_campaign

---

## Question 5

In [6]:
order_items = pd.read_csv('order_items.csv', low_memory=False)
percent_no_promo = order_items['promo_id'].isnull().sum()/len(order_items)
percent_promo = (1 - percent_no_promo)*100

print(f'Tỷ lệ phần trăm các dòng có áp dụng khuyến mãi: {percent_promo:.0f}%')

Tỷ lệ phần trăm các dòng có áp dụng khuyến mãi: 39%


Đáp án: C) 39%

---

## Question 6

In [7]:
customers = pd.read_csv('customers.csv')
orders_per_customer = orders.groupby('customer_id').size()
customers = customers.merge(orders_per_customer.rename('order_count'), on='customer_id', how='left')
customers['order_count'] = customers['order_count'].fillna(0)
customers_w_age = customers[customers['age_group'].notnull()]
avg_orders_per_age = customers_w_age.groupby('age_group')['order_count'].mean()

print(avg_orders_per_age)
print('-'*40)
print(f'Nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất: {avg_orders_per_age.idxmax()}')

age_group
18-24    5.226656
25-34    5.245226
35-44    5.337343
45-54    5.357241
55+      5.406851
Name: order_count, dtype: float64
----------------------------------------
Nhóm tuổi có số đơn hàng trung bình trên mỗi khách hàng cao nhất: 55+


Đáp án: A) 55+

---

## Question 7 (solved by Tableau)

---

## Question 8

In [8]:
order_cancelled = orders[orders['order_status']=='cancelled'].copy()
cancel_method = order_cancelled.groupby('payment_method').size()

print(cancel_method)
print('-'*40)
print(f'Phương thức thanh toán có nhiều đơn bị huỷ nhất: {cancel_method.idxmax()}')

payment_method
apple_pay         5190
bank_transfer     2535
cod              15468
credit_card      28452
paypal            7817
dtype: int64
----------------------------------------
Phương thức thanh toán có nhiều đơn bị huỷ nhất: credit_card


Đáp án: A) credit_card

---

## Question 9

In [9]:
ret_p = returns.merge(products[['product_id','size']], on='product_id', how='left')
# Denominator: order_items × products
items_p = order_items.merge(products[['product_id','size']], on='product_id', how='left')

ratio = ret_p['size'].value_counts() / items_p['size'].value_counts()

print(ratio)
print('-'*40)
print(f'Kích thước có tỉ lệ trả hàng cao nhất: {ratio.idxmax()}')

size
XL    0.055200
M     0.055660
L     0.056250
S     0.056515
Name: count, dtype: float64
----------------------------------------
Kích thước có tỉ lệ trả hàng cao nhất: S


Đáp án: A) S

---

## Question 10

In [10]:
payments = pd.read_csv('payments.csv')
mean_payment_value = payments.groupby('installments')['payment_value'].mean()

print(mean_payment_value)
print('-'*40)
print(f'Kế hoạch trả góp có giá trị thanh toán trung bình trên mỗi đơn cao nhất: {mean_payment_value.idxmax()}')

installments
1     24113.274166
2       708.473729
3     24399.635486
6     24446.654403
12    24245.772694
Name: payment_value, dtype: float64
----------------------------------------
Kế hoạch trả góp có giá trị thanh toán trung bình trên mỗi đơn cao nhất: 6


Đáp án: C) 6 kỳ